In [3]:
import pandas as pd

url = "https://baseball-data.com/25/lineup/g.html"

tables = pd.read_html(url)
print("table数:", len(tables))

for i, df in enumerate(tables):
    cols = [str(c).strip() for c in df.columns]
    print(f"\n--- table {i} ---")
    print(cols[:15])
    
    if "日付" in cols and "4番" in cols:
        target = df.copy()
        print("\n見つかった表:")
        print(target.head())
        break
else:
    raise ValueError("『日付』『4番』を含む表が見つかりませんでした")

table数: 1

--- table 0 ---
['日付', '1番', '2番', '3番', '4番', '5番', '6番', '7番', '8番', '9番']

見つかった表:
         日付      1番     2番     3番     4番      5番     6番     7番    8番     9番
0  3月28日(金)   若林 楽人  キャベッジ  吉川 尚輝  岡本 和真  ヘルナンデス  坂本 勇人  甲斐 拓也  門脇 誠  戸郷 翔征
1  3月29日(土)   若林 楽人  キャベッジ  吉川 尚輝  岡本 和真  ヘルナンデス  中山 礼都  甲斐 拓也  門脇 誠  赤星 優志
2  3月30日(日)   若林 楽人  キャベッジ  吉川 尚輝  岡本 和真  ヘルナンデス  坂本 勇人  甲斐 拓也  門脇 誠  石川 達也
3   4月1日(火)  オコエ 瑠偉  キャベッジ  吉川 尚輝  岡本 和真  ヘルナンデス  甲斐 拓也  中山 礼都  門脇 誠  井上 温大
4   4月2日(水)   中山 礼都  キャベッジ  吉川 尚輝  岡本 和真  ヘルナンデス  甲斐 拓也  萩尾 匡也  門脇 誠  山﨑 伊織


In [5]:
import pandas as pd
import time

TEAM_URLS = {
    "阪神": "https://baseball-data.com/25/lineup/t.html",
    "DeNA": "https://baseball-data.com/25/lineup/yb.html",
    "巨人": "https://baseball-data.com/25/lineup/g.html",
    "中日": "https://baseball-data.com/25/lineup/d.html",
    "広島": "https://baseball-data.com/25/lineup/c.html",
    "ヤクルト": "https://baseball-data.com/25/lineup/s.html",
    "ソフトバンク": "https://baseball-data.com/25/lineup/h.html",
    "日本ハム": "https://baseball-data.com/25/lineup/f.html",
    "オリックス": "https://baseball-data.com/25/lineup/bs.html",  # ←ここを修正
    "楽天": "https://baseball-data.com/25/lineup/e.html",
    "西武": "https://baseball-data.com/25/lineup/l.html",
    "ロッテ": "https://baseball-data.com/25/lineup/m.html",
}

def load_one_team(team_name, url):
    print(f"\n取得中: {team_name}")
    tables = pd.read_html(url)

    target = None
    for df in tables:
        df = df.copy()
        df.columns = [str(c).strip() for c in df.columns]
        cols = list(df.columns)
        if "日付" in cols and "4番" in cols:
            target = df
            break

    if target is None:
        raise ValueError(f"{team_name}: 『日付』『4番』を含む表が見つかりません")

    keep_cols = ["日付", "1番", "2番", "3番", "4番", "5番", "6番", "7番", "8番", "9番"]
    target = target[[c for c in keep_cols if c in target.columns]].copy()

    target = target[target["日付"].astype(str).str.strip() != "日付"].copy()
    target = target.dropna(subset=["日付", "4番"]).copy()

    for col in keep_cols[1:]:
        if col in target.columns:
            target[col] = target[col].astype(str).str.strip()

    target.insert(0, "球団", team_name)
    print(f"{team_name}: {len(target)}行")
    return target

all_dfs = []
failed = []

for team, url in TEAM_URLS.items():
    try:
        df = load_one_team(team, url)
        all_dfs.append(df)
        time.sleep(1)
    except Exception as e:
        print("失敗:", e)
        failed.append((team, str(e)))

if not all_dfs:
    raise RuntimeError("全球団失敗。まずは巨人1球団コードが通るか確認してください。")

lineup_df = pd.concat(all_dfs, ignore_index=True)
lineup_df.to_csv("npb_2025_lineup_all.csv", index=False, encoding="utf-8-sig")

fourth_count_df = (
    lineup_df.groupby(["球団", "4番"])
    .size()
    .reset_index(name="4番出場回数")
    .sort_values(["球団", "4番出場回数"], ascending=[True, False])
)
fourth_count_df.to_csv("npb_2025_fourth_count.csv", index=False, encoding="utf-8-sig")

main_fourth_df = (
    fourth_count_df.sort_values(["球団", "4番出場回数", "4番"], ascending=[True, False, True])
    .groupby("球団", as_index=False)
    .first()
)
main_fourth_df.to_csv("npb_2025_main_fourth_batters.csv", index=False, encoding="utf-8-sig")

print("\n=== 各球団の代表4番打者 ===")
print(main_fourth_df)

if failed:
    print("\n=== 失敗した球団 ===")
    for x in failed:
        print(x)


取得中: 阪神
阪神: 143行

取得中: DeNA
DeNA: 143行

取得中: 巨人
巨人: 143行

取得中: 中日
中日: 143行

取得中: 広島
広島: 143行

取得中: ヤクルト
ヤクルト: 143行

取得中: ソフトバンク
ソフトバンク: 143行

取得中: 日本ハム
日本ハム: 143行

取得中: オリックス
オリックス: 143行

取得中: 楽天
楽天: 143行

取得中: 西武
西武: 143行

取得中: ロッテ
ロッテ: 143行

=== 各球団の代表4番打者 ===
        球団      4番  4番出場回数
0     DeNA    牧 秀悟      60
1    オリックス  杉本 裕太郎      74
2   ソフトバンク   山川 穂高      68
3     ヤクルト     オスナ      64
4      ロッテ   山本 大斗      47
5       中日   細川 成也      75
6       巨人   岡本 和真      66
7       広島   末包 昇大      63
8     日本ハム   野村 佑希      52
9       楽天     ボイト      35
10      西武     ネビン     119
11      阪神   佐藤 輝明     126
